In [1]:
# file /public/trendytech/retail_db/orders/part-00000

##### RDD  - map, reduceByKey , sortBy

In [2]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [3]:
rdd1 = spark.sparkContext.textFile("/public/trendytech/retail_db/orders/part-00000")

In [4]:
rdd1.take(5)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT',
 '3,2013-07-25 00:00:00.0,12111,COMPLETE',
 '4,2013-07-25 00:00:00.0,8827,CLOSED',
 '5,2013-07-25 00:00:00.0,11318,COMPLETE']

In [5]:
rdd2 = rdd1.map(lambda x: (x.split(",")))

In [6]:
rdd2.take(5)

[['1', '2013-07-25 00:00:00.0', '11599', 'CLOSED'],
 ['2', '2013-07-25 00:00:00.0', '256', 'PENDING_PAYMENT'],
 ['3', '2013-07-25 00:00:00.0', '12111', 'COMPLETE'],
 ['4', '2013-07-25 00:00:00.0', '8827', 'CLOSED'],
 ['5', '2013-07-25 00:00:00.0', '11318', 'COMPLETE']]

In [7]:
rdd2 = rdd1.map(lambda x: (x.split(",")[3],1))

In [8]:
rdd2.take(5)

[('CLOSED', 1),
 ('PENDING_PAYMENT', 1),
 ('COMPLETE', 1),
 ('CLOSED', 1),
 ('COMPLETE', 1)]

In [9]:
rdd3 = rdd2.reduceByKey(lambda x,y : x+y)

In [10]:
rdd3.take(5)

[('CLOSED', 7556),
 ('CANCELED', 1428),
 ('PENDING_PAYMENT', 15030),
 ('COMPLETE', 22899),
 ('PROCESSING', 8275)]

In [11]:
rdd4 = rdd3.sortBy(lambda x:x[1],False)

In [12]:
rdd4.take(50)

[('COMPLETE', 22899),
 ('PENDING_PAYMENT', 15030),
 ('PROCESSING', 8275),
 ('PENDING', 7610),
 ('CLOSED', 7556),
 ('ON_HOLD', 3798),
 ('SUSPECTED_FRAUD', 1558),
 ('CANCELED', 1428),
 ('PAYMENT_REVIEW', 729)]

####### Count of records grouped by Order status 

In [13]:
rdd1.take(5)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT',
 '3,2013-07-25 00:00:00.0,12111,COMPLETE',
 '4,2013-07-25 00:00:00.0,8827,CLOSED',
 '5,2013-07-25 00:00:00.0,11318,COMPLETE']

In [14]:
rdd5 = rdd1.map(lambda x: (x.split(",")[2],1))

In [15]:
rdd5.take(5)

[('11599', 1), ('256', 1), ('12111', 1), ('8827', 1), ('11318', 1)]

In [16]:
rdd6 = rdd5.reduceByKey(lambda x,y: x+y)

In [17]:
rdd6.take(5)

[('256', 10), ('12111', 6), ('11318', 6), ('7130', 7), ('2911', 6)]

In [18]:
rdd7 = rdd6.sortBy(lambda x:x[1],False)

####### Count of records grouped by Customer ID

In [19]:
rdd7.take(5)

[('5897', 16), ('569', 16), ('6316', 16), ('12431', 16), ('4320', 15)]

##### Distinct

In [28]:
rdd1.count()

68883

In [20]:
rdd8 = rdd1.map(lambda x: (x.split(",")[2]))

In [21]:
rdd8.take(5)

['11599', '256', '12111', '8827', '11318']

In [26]:
dist_cust = rdd1.map(lambda x: (x.split(",")[2])).distinct()

In [27]:
dist_cust.count()

12405

##### customer with max closed orders

In [29]:
closed_cust = rdd1.filter(lambda x: (x.split(",")[3]=='CLOSED'))

In [30]:
closed_cust.take(5)

['1,2013-07-25 00:00:00.0,11599,CLOSED',
 '4,2013-07-25 00:00:00.0,8827,CLOSED',
 '12,2013-07-25 00:00:00.0,1837,CLOSED',
 '18,2013-07-25 00:00:00.0,1205,CLOSED',
 '24,2013-07-25 00:00:00.0,11441,CLOSED']

In [37]:
max_closed = closed_cust.map(lambda x: (x.split(",")[2],1))

In [38]:
max_closed.take(5)

[('11599', 1), ('8827', 1), ('1837', 1), ('1205', 1), ('11441', 1)]

In [45]:
max_closed_dist = max_closed.reduceByKey(lambda x,y : x+y ).sortBy(lambda x: x[1],False)

In [46]:
max_closed_dist.take(5)

[('1833', 6), ('5493', 5), ('1363', 5), ('1687', 5), ('2768', 4)]